In [1]:
!uv pip install -U Ipython
!uv pip install webvtt-py sacrebleu rouge_score pycocoevalcap
!uv pip install peft hf_xet hf-trim evaluate transformers==4.57.3
!pip install git+https://github.com/google-research/bleurt.git

Using Python 3.12.13 environment at: /usr
Resolved 17 packages in 168ms
Prepared 12 packages in 660ms
Uninstalled 6 packages in 28ms
Installed 12 packages in 138ms
 + asttokens==3.0.1
 - decorator==4.4.2
 + decorator==5.2.1
 + executing==2.2.1
 - ipython==7.34.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.20.0
 - parso==0.8.6
 + parso==0.8.7
 - psutil==5.9.5
 + psutil==7.2.2
 + pure-eval==0.2.3
 + stack-data==0.6.3
 - traitlets==5.7.1
 + traitlets==5.14.3
 - wcwidth==0.6.0
 + wcwidth==0.7.0
Using Python 3.12.13 environment at: /usr
Resolved 17 packages in 863ms
Prepared 6 packages in 411ms
Installed 6 packages in 3ms
 + colorama==0.4.6
 + portalocker==3.2.0
 + pycocoevalcap==1.2
 + rouge-score==0.1.2
 + sacrebleu==2.6.0
 + webvtt-py==0.5.1
Using Python 3.12.13 environment at: /usr
Resolved 68 packages in 641ms
Prepared 4 packages in 421ms
Uninstalled 2 packages in 349ms
Installed 4 packages in 58ms
 + evaluate==0.4.6
 + hf-trim==3.0.1
 - huggingface-hub==1.11.0
 + h

In [2]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/e2e-slt-streaming
%load_ext autoreload
%autoreload 2

# !wget https://storage.googleapis.com/bleurt-oss-21/BLEURT-20.zip -O ./checkpoints/BLEURT-20.zip
# !git fetch origin && git reset --hard origin/main # Don't care about local changes and overwrite everything
# !git pull
!ls

Mounted at /content/drive/
/content/drive/.shortcut-targets-by-id/1AlNQaxmBldq_vhxtUYkHpWaALnGeTKc4/e2e-slt-streaming
backbones	 docs			 gfslt_stage2.py  postprocess.py
captioners	 eval.py		 __init__.py	  __pycache__
checkpoints	 evaluation		 loader.py	  README.md
config.py	 experiments		 loss.py	  requirements.txt
data		 gfslt_cascaded_eval.py  main.py	  test.py
data_synth	 gfslt_models.py	 pdvc.py	  utils.py
deformable_detr  gfslt_stage1.py	 poses


In [3]:
%%time
%env DATASET = phoenix
!rm -rf /tmp/"${DATASET}"
!7z x ./data/synth/"${DATASET}.zip" -o/tmp
!7z x ./checkpoints/BLEURT-20.zip -o/tmp

env: DATASET=phoenix

7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,12 CPUs Intel(R) Xeon(R) CPU @ 2.20GHz (50657),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan ./data/synth/                       1 file, 389049542 bytes (372 MiB)

Extracting archive: ./data/synth/phoenix.zip
--
Path = ./data/synth/phoenix.zip
Type = zip
Physical Size = 389049542

  0%      3% 49 - phoenix/poses/train_00007.npy                                         7% 70 - phoenix/poses/train_00028.npy                                        10% 88 - phoenix/poses/train_00046.npy                                        14% 112 - phoenix

# SreamSLST [LSTM]

In [ ]:
%%time
!python3 main.py \
	--mode 1 \
	--output_dir /tmp/streamslst_mode1 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--num_train_epochs 64 \
	--captioner_type lstm \
	--learning_rate 5e-4 \
    --weight_decay 0.05 \
	--per_device_train_batch_size 32

2026-05-03 19:53:13.747946: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 19:53:13.779439: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 5992.41it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)

Training Mode: 1
Train dataset: 564 samples

MODE 1: Contrastive Pre-training (backbone + text encoder trainable)
Trainable parameters: 12.02M / 133.05

In [ ]:
%%time
!python3 main.py \
	--mode 2 \
	--mode1_checkpoint /tmp/streamslst_mode1/mode1_final \
	--output_dir /tmp/streamslst_mode2 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--num_train_epochs 100 \
	--captioner_type lstm \
	--per_device_train_batch_size 32 \
	--learning_rate 2e-4
!cp -r /tmp/streamslst_mode2/mode2_final ./checkpoints/"${DATASET}_lstm"

2026-05-03 20:05:28.317516: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 20:05:28.349739: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 6191.84it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)

Training Mode: 2
Train dataset: 564 samples
Loading mode 1 (contrastive) checkpoint from: /tmp/streamslst_mode1/mode1_final/pytorch_model.bin
=> Skippi

In [14]:
%%time
!python eval.py \
	--checkpoint_path ./checkpoints/"${DATASET}_lstm" \
	--eval_val False \
	--per_device_eval_batch_size 128 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--captioner_type lstm

2026-05-04 02:26:28.369274: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:26:28.442817: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 02:26:39.056834: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777861599.057838   30795 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam

# SreamSLST [mBART]

In [ ]:
%%time
!python3 main.py \
	--mode 1 \
	--output_dir /tmp/streamslst_mode1 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--num_train_epochs 64 \
	--captioner_type mbart \
	--learning_rate 5e-4 \
    --weight_decay 0.05 \
	--per_device_train_batch_size 32

2026-05-03 19:53:14.174810: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 19:53:14.205862: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 5919.06it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)

Training Mode: 1
Train dataset: 564 samples

MODE 1: Contrastive Pre-training (backbone + text encoder trainable)
Trainable parameters: 12.02M / 163.96

In [ ]:
%%time
!python3 main.py \
	--mode 2 \
	--mode1_checkpoint /tmp/streamslst_mode1/mode1_final \
	--output_dir /tmp/streamslst_mode2 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--num_train_epochs 100 \
	--captioner_type mbart \
	--per_device_train_batch_size 32 \
	--learning_rate 2e-4
!cp -r /tmp/streamslst_mode2/mode2_final ./checkpoints/"${DATASET}_mbart"

2026-05-03 20:00:50.756807: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 20:00:50.788947: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 6167.85it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)

Training Mode: 2
Train dataset: 564 samples
Loading mode 1 (contrastive) checkpoint from: /tmp/streamslst_mode1/mode1_final/pytorch_model.bin
=> Skippi

In [15]:
%%time
!python eval.py \
	--checkpoint_path ./checkpoints/"${DATASET}_mbart" \
	--eval_val False \
	--per_device_eval_batch_size 64 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--captioner_type mbart

2026-05-04 02:27:37.840619: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:27:37.911100: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 02:27:48.859929: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777861668.861068   31335 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam

# Multi-Stage [Cascaded]

In [ ]:
%%time
!python3 gfslt_stage1.py \
	--output_dir /tmp/gfslt_stage1 \
	--max_event_tokens 40 \
	--max_window_tokens 128 \
	--embed_dim 1024 \
	--num_train_epochs 80 \
	--learning_rate 0.01 \
    --weight_decay 0.0 \
	--per_device_train_batch_size 128

2026-05-03 21:44:26.576943: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 21:44:26.650747: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 1913.05it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)
Train dataset: 564 samples
Number of trainable parameters: 101.11M
{'loss': 28.0495, 'grad_norm': 0.058266133069992065, 'learning_rate': 0.009997532801828659, 'epoch': 1.0}


In [ ]:
%%time
!python3 gfslt_stage2.py \
	--output_dir /tmp/gfslt_stage2 \
    --stage1_checkpoint /tmp/gfslt_stage1/pytorch_model.bin \
	--max_event_tokens 40 \
	--max_window_tokens 128 \
	--embed_dim 1024 \
	--num_train_epochs 200 \
	--learning_rate 0.001 \
    --weight_decay 1e-3 \
	--per_device_train_batch_size 256 \
	--do_train
!cp -r /tmp/gfslt_stage2 ./checkpoints/"${DATASET}_gfslt"

2026-05-03 22:32:22.545141: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 22:32:22.621394: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-03 22:32:31.845426: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777847551.846565   57180 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam

In [16]:
%%time
!python3 gfslt_stage2.py \
	--output_dir ./checkpoints/"${DATASET}_gfslt" \
	--max_event_tokens 40 \
	--max_window_tokens 128 \
	--embed_dim 1024 \
	--num_train_epochs 200 \
	--learning_rate 0.001 \
    --weight_decay 1e-3 \
	--per_device_eval_batch_size 1024 \
	--do_eval

2026-05-04 02:28:28.629324: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:28:28.702666: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 02:28:37.586973: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777861717.588100   31779 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam

In [17]:
%%time
!python gfslt_cascaded_eval.py \
    --detr_checkpoint ./checkpoints/"${DATASET}_mbart"/pytorch_model.bin \
    --gfslt_checkpoint ./checkpoints/"${DATASET}_gfslt"/pytorch_model.bin \
    --per_device_eval_batch_size 128 \
    --gfslt_max_new_tokens 40 \
    --eval_test

2026-05-04 02:29:32.982095: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:29:33.053569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 02:29:43.512821: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777861783.513811   32233 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam

# Abl Study 1: StreamSLST[mBART] without VLP

In [9]:
%%time
!python3 main.py \
	--mode 2 \
	--output_dir /tmp/streamslst_mode2 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--num_train_epochs 100 \
	--captioner_type mbart \
	--per_device_train_batch_size 32 \
	--learning_rate 5e-4
!cp -r /tmp/streamslst_mode2/mode2_final ./checkpoints/"${DATASET}_abl1"

2026-05-04 01:50:05.813976: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 01:50:05.884428: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 1940.11it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)

Training Mode: 2
Train dataset: 564 samples

MODE 2: Joint Training (all parameters trainable)
Trainable parameters: 161.14M / 161.14M (100.0%)
{'loss': 54.0323, 'grad_norm

In [10]:
%%time
!python eval.py \
	--checkpoint_path ./checkpoints/"${DATASET}_abl1" \
	--eval_val False \
	--per_device_eval_batch_size 64 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
	--captioner_type mbart

2026-05-04 02:08:22.628456: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:08:22.699897: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 02:08:33.533450: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777860513.534580   18975 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam

# Abl Study 2: StreamSLST[mBART] without Learnt Proposals

In [12]:
%%time
!python3 main.py \
	--mode 2 \
	--output_dir /tmp/streamslst_mode2 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
    --with_box_refine False \
	--num_train_epochs 100 \
	--captioner_type mbart \
	--per_device_train_batch_size 32 \
	--learning_rate 5e-4
!cp -r /tmp/streamslst_mode2/mode2_final ./checkpoints/"${DATASET}_abl2"

2026-05-04 02:11:15.459347: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:11:15.532654: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Found 564 videos in the train split.
Building video metadata for train split: 100% 564/564 [00:00<00:00, 1971.04it/s]
Dataset initialized for train: 564 videos
Window size: 15s (187 frames @ 12.5 fps)

Training Mode: 2
Train dataset: 564 samples

MODE 2: Joint Training (all parameters trainable)
Trainable parameters: 105.75M / 105.75M (100.0%)
{'loss': 33.0466, 'grad_norm

In [13]:
%%time
!python eval.py \
	--checkpoint_path ./checkpoints/"${DATASET}_abl2" \
	--eval_val False \
	--per_device_eval_batch_size 64 \
	--max_event_tokens 40 \
	--d_model 1024 \
	--encoder_layers 2 \
	--decoder_layers 2 \
	--num_cap_layers 3 \
	--num_queries 30 \
    --with_box_refine False \
	--captioner_type mbart

2026-05-04 02:25:19.462052: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-04 02:25:19.533972: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 02:25:30.420817: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777861530.421948   30308 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79188 MB memory:  -> device: 0, nam